# Домашнее задание 3

## Задание 1.2. Параметризованные запросы

In [ ]:
%pip install -q mysql-connector-python pandas

In [ ]:
import os
import mysql.connector
import pandas as pd

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "127.0.0.1"),
    "port": int(os.getenv("DB_PORT", "3306")),
    "user": "root",
    "password": os.getenv("MYSQL_ROOT_PASSWORD", "chinook_root"),
    "database": "Chinook_AutoIncrement",
}

con = mysql.connector.connect(**DB_CONFIG)
con.autocommit = True
print("Подключено:", con.is_connected(), "| MySQL", con.get_server_info())


def query(sql, params=()):
    cur = con.cursor()
    try:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=cur.column_names)
    finally:
        cur.close()

### 1. Все треки заданного жанра

In [ ]:
def tracks_by_genre(genre):
    sql = """
        SELECT t.TrackId, t.Name AS Track, al.Title AS Album, ar.Name AS Artist, g.Name AS Genre
        FROM Track t
        JOIN Genre g ON g.GenreId = t.GenreId
        LEFT JOIN Album al ON al.AlbumId = t.AlbumId
        LEFT JOIN Artist ar ON ar.ArtistId = al.ArtistId
        WHERE g.Name = %s
        ORDER BY t.TrackId
    """
    return query(sql, (genre,))


rock_tracks = tracks_by_genre("Rock")
print("Треков в жанре Rock:", len(rock_tracks))
rock_tracks.head(10)

### 2. Все альбомы артиста по имени

In [ ]:
def albums_by_artist(artist_name):
    sql = """
        SELECT ar.Name AS Artist, al.AlbumId, al.Title
        FROM Album al
        JOIN Artist ar ON ar.ArtistId = al.ArtistId
        WHERE ar.Name = %s
        ORDER BY al.Title
    """
    return query(sql, (artist_name,))


display(albums_by_artist("AC/DC"))
display(albums_by_artist("Iron Maiden"))

### 3. Отчёт по клиенту

In [ ]:
def customer_report(customer_id):
    sql = """
        SELECT c.CustomerId, c.FirstName, c.LastName, c.Email,
               (SELECT COUNT(*) FROM Invoice i WHERE i.CustomerId = c.CustomerId) AS InvoiceCount,
               (SELECT COALESCE(SUM(i.Total), 0) FROM Invoice i WHERE i.CustomerId = c.CustomerId) AS TotalSpent,
               (SELECT g.Name
                FROM Invoice i
                JOIN InvoiceLine il ON il.InvoiceId = i.InvoiceId
                JOIN Track t ON t.TrackId = il.TrackId
                JOIN Genre g ON g.GenreId = t.GenreId
                WHERE i.CustomerId = c.CustomerId
                GROUP BY g.GenreId, g.Name
                ORDER BY SUM(il.Quantity) DESC, g.Name
                LIMIT 1) AS TopGenre
        FROM Customer c
        WHERE c.CustomerId = %s
    """
    return query(sql, (customer_id,))


customer_report(1)

### 4. Треки, в названии которых есть подстрока

In [ ]:
def tracks_by_substring(substring):
    sql = """
        SELECT t.TrackId, t.Name AS Track, g.Name AS Genre
        FROM Track t
        LEFT JOIN Genre g ON g.GenreId = t.GenreId
        WHERE t.Name LIKE %s
        ORDER BY t.Name
    """
    return query(sql, (f"%{substring}%",))


love_tracks = tracks_by_substring("love")
print("Найдено треков:", len(love_tracks))
love_tracks.head(10)

## Задание 1.3. Транзакции и модификация данных

### 1. Таблица customer_feedback

In [ ]:
cur = con.cursor()
cur.execute("DROP TABLE IF EXISTS customer_feedback")
cur.execute("""
    CREATE TABLE customer_feedback (
        feedback_id INT AUTO_INCREMENT PRIMARY KEY,
        customer_id INT NOT NULL,
        track_id INT NOT NULL,
        rating INT NOT NULL,
        comment TEXT NULL,
        created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
        CONSTRAINT fk_feedback_customer FOREIGN KEY (customer_id) REFERENCES Customer (CustomerId),
        CONSTRAINT fk_feedback_track FOREIGN KEY (track_id) REFERENCES Track (TrackId),
        CONSTRAINT chk_feedback_rating CHECK (rating BETWEEN 1 AND 5)
    )
""")
cur.execute("SHOW CREATE TABLE customer_feedback")
print(cur.fetchone()[1])
cur.close()

### 2. Вставка записей через executemany() в одной транзакции

In [ ]:
INSERT_FEEDBACK = """
    INSERT INTO customer_feedback (customer_id, track_id, rating, comment)
    VALUES (%s, %s, %s, %s)
"""

feedback_rows = [
    (1, 1, 5, "Классика рока"),
    (2, 2, 4, "Отличный рифф"),
    (3, 6, 3, None),
    (4, 15, 1, "Не моё"),
    (5, 63, 2, "Слишком спокойно"),
    (6, 77, 4, "Мощно"),
]

con.start_transaction()
cur = con.cursor()
try:
    cur.executemany(INSERT_FEEDBACK, feedback_rows)
    con.commit()
    print("COMMIT, вставлено строк:", cur.rowcount)
except mysql.connector.Error as e:
    con.rollback()
    print("ROLLBACK:", e)
finally:
    cur.close()

query("SELECT * FROM customer_feedback ORDER BY feedback_id")

### 3. Пакетное обновление оценок для жанра в одной транзакции

In [ ]:
def feedback_with_genre():
    return query("""
        SELECT cf.feedback_id, cf.track_id, t.Name AS Track, g.Name AS Genre, cf.rating
        FROM customer_feedback cf
        JOIN Track t ON t.TrackId = cf.track_id
        JOIN Genre g ON g.GenreId = t.GenreId
        ORDER BY cf.feedback_id
    """)


genre = "Rock"
before = feedback_with_genre()

con.start_transaction()
cur = con.cursor()
try:
    cur.execute("""
        UPDATE customer_feedback cf
        JOIN Track t ON t.TrackId = cf.track_id
        JOIN Genre g ON g.GenreId = t.GenreId
        SET cf.rating = LEAST(cf.rating + 1, 5)
        WHERE g.Name = %s
    """, (genre,))
    changed = cur.rowcount
    con.commit()
    print(f"COMMIT, изменено строк для жанра {genre}:", changed)
except mysql.connector.Error as e:
    con.rollback()
    print("ROLLBACK:", e)
finally:
    cur.close()

after = feedback_with_genre()
before.merge(after[["feedback_id", "rating"]], on="feedback_id", suffixes=("_before", "_after"))

### 4. Откат транзакции при ошибке

In [ ]:
marker = "rollback demo"
print("Записей до транзакции:", query("SELECT COUNT(*) AS cnt FROM customer_feedback").iloc[0, 0])

con.start_transaction()
cur = con.cursor()
try:
    cur.execute(INSERT_FEEDBACK, (7, 3000, 5, marker))
    print("Валидная запись вставлена внутри транзакции, feedback_id =", cur.lastrowid)
    cur.execute(INSERT_FEEDBACK, (8, 3000, 10, marker))
    con.commit()
    print("COMMIT")
except mysql.connector.Error as e:
    con.rollback()
    print("Ошибка:", e)
    print("Выполнен ROLLBACK")
finally:
    cur.close()

print("Записей после транзакции:", query("SELECT COUNT(*) AS cnt FROM customer_feedback").iloc[0, 0])
query("SELECT * FROM customer_feedback WHERE comment = %s", (marker,))

In [ ]:
con.close()